# 00 · Setup & environment check

**Run this notebook before the tutorial.** It is a pass/fail check: if the last cell draws a bar chart with two bars (`00` and `11`, each around 500 counts), you are ready.

Any Jupyter environment works: JupyterLab, VS Code, or Google Colab.

- **Local:** `pip install "qilisdk==0.3.0"` (Python 3.11–3.14), then run the cells below.
- **Colab:** just run the cells top to bottom. The install cell fetches QiliSDK for you.

QiliSDK ships its C++ simulator, **QiliSim**, *inside* the wheel, so a plain `pip install qilisdk` gives you everything needed to simulate quantum programs locally: no compiler, no GPU, and no cloud account required.

One term you will meet immediately: a **qubit** is the quantum version of a bit. Part 1 begins there.

## What to expect today

The tutorial has two parts, and each is built around a problem you could plausibly meet at work. By the end of the session you will have:

- Built quantum states from scratch and turned the simplest one into a **true random number generator**, the same recipe commercial QRNG chips use (Parts 1 and 2).
- Scaled that generator into a **provably fair dice roller**, and used entanglement to give **three parties a shared random secret** (Part 2).
- **Compiled a circuit onto a real chip's layout** and counted what the chip's wiring costs you (Part 2).

Practical notes:

- Bring a laptop. Everything runs on an ordinary CPU, no GPU or cloud account needed.
- The session is 90 minutes, so please complete this setup check before you arrive.
- If you get stuck during an exercise, the completed notebooks live one folder away in `notebooks/solutions/`.

### Check 1: Python version

QiliSDK 0.3.0 ships pre-compiled C++ extension modules, so the interpreter version matters the same way it does for numpy wheels: the package supports **Python 3.11 to 3.14**. This cell tells you immediately whether your kernel qualifies, and what to do if it does not.

In [ ]:
import sys

version = ".".join(str(v) for v in sys.version_info[:3])
if (3, 11) <= sys.version_info[:2] <= (3, 14):
    print(f"OK: Python {version} is supported.")
else:
    print(f"Python {version} is NOT supported. QiliSDK 0.3.0 needs Python 3.11, 3.12, 3.13, or 3.14.")
    print("- On Google Colab: Runtime > Change runtime type, then pick a supported Python version.")
    print("- Locally: create a virtual environment with a supported Python, for example:")
    print("      python3.11 -m venv qilisdk-env")
    print("      source qilisdk-env/bin/activate")
    print("  then install the requirements and reopen this notebook from that environment.")

### Check 2: install QiliSDK

The cell below is a no-op if QiliSDK is already installed, and installs it otherwise (a fresh Colab runtime, for example). We pin `qilisdk==0.3.0` because that is the exact version every example in this tutorial was verified against. If a different QiliSDK version is already installed, the cell tells you how to switch.

In [ ]:
# ▶ No-op if QiliSDK is already installed; installs it on a fresh env (e.g. Google Colab).
try:
    import qilisdk
except ImportError:
    import sys
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "qilisdk==0.3.0"], check=True)
    import qilisdk  # Colab: if this still fails, Runtime > Restart session, then rerun
print("QiliSDK", qilisdk.__version__)
if qilisdk.__version__ != "0.3.0":
    print('This tutorial was verified with QiliSDK 0.3.0. To match it, run: pip install "qilisdk==0.3.0"')

### Check 3: full diagnostics

`qilisdk.about()` prints a report of your environment: the QiliSDK version, your Python / NumPy / SciPy versions, your CPU/GPU, and, crucially, whether the compiled **QiliSim** simulator and the **QTensor** core loaded successfully. If those two say `Success`, the C++ extension modules are working on your machine.

(`about()` *returns* a string rather than printing it, so wrap it in `print`.)

**If QiliSim or QTensor does not say `Success`**, work through these in order:

1. Reinstall into a fresh virtual environment (each command on its own line):

   ```
   python3.11 -m venv qilisdk-env
   source qilisdk-env/bin/activate
   pip install "qilisdk==0.3.0" notebook
   ```

   On Windows, activate with `qilisdk-env\Scripts\activate` instead of the `source` line.
2. If a local install keeps failing, fall back to **Google Colab**: upload this notebook and run it there, the install cell above handles the rest.
3. Still stuck? Arrive 15 minutes early on tutorial day or contact the instructor, and we will fix it together.

In [ ]:
import qilisdk

print(qilisdk.about())

### Check 4: your first quantum program

This builds a 2-qubit **Bell state**, simulates it, and measures it 1000 times. You will understand every line by the end of Part 2; for now we just want to confirm the whole stack (build, then simulate, then measure) runs end to end. The four API names, one line each:

- `QiliSim` is the simulator: the engine that crunches the math on your CPU.
- `DigitalPropagation(circuit)` means "run this circuit" (a bare circuit is a description, this wraps it into something executable).
- `Readout()` declares what to record; here, 1000 measurement shots.
- `M` is the measurement instruction: it turns quantum state into classical bits you can read.

The expected result is a **perfectly correlated** outcome: the two qubits always agree, so you see only `00` and `11`, each about half the time, and essentially never `01` or `10`. That correlation is *entanglement*, and reproducing it is the "hello world" of quantum computing.

In [ ]:
from qilisdk.digital import Circuit, H, CNOT, M
from qilisdk.functionals import DigitalPropagation
from qilisdk.backends import QiliSim
from qilisdk.readout import Readout

circuit = Circuit(2)
circuit.add(H(0))          # build: put qubit 0 into superposition
circuit.add(CNOT(0, 1))    # build: entangle qubit 0 with qubit 1
circuit.add(M(0, 1))       # build: measure both qubits

# simulate and measure: 1000 shots on the QiliSim simulator
result = QiliSim().execute(DigitalPropagation(circuit), Readout().with_sampling(nshots=1000))
print(result.get_samples())

If you see roughly `{'00': ~500, '11': ~500}` (the exact split varies run to run, since these are genuine random samples), your environment simulates quantum programs correctly.

Why care about two random bits that always agree? Perfectly correlated random bits shared by two parties are the raw ingredient of quantum-secure key exchange, and the H-plus-measure pattern inside this circuit is essentially how commercial quantum random number generators work (Part 2 builds one).

### Check 5: plotting

Both parts of the tutorial draw figures (Bloch spheres, circuit diagrams, histograms), so the last check is matplotlib itself: turn the counts you just measured into a bar chart. If you see a picture, plotting works.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

counts = dict(sorted(result.get_samples().items()))
plt.bar(list(counts.keys()), list(counts.values()))
plt.xlabel("measured bitstring")
plt.ylabel("counts out of 1000 shots")
plt.title("Bell state: the two qubits always agree")
plt.show()

Two bars of roughly equal height, nothing on `01` or `10`: everything works. See you at the tutorial!

Curious already? Change `nshots=1000` to `20` in Check 4 and rerun that cell and the plot: the split gets much noisier (a 13 to 7 or even 4 to 16 split is normal at 20 shots). Quantum results are statistics, and the shot count is the knob that trades runtime for precision. Part 2 makes heavy use of that knob.